In [ ]:
# Install the PDF reader and the Google GenAI SDK
!pip install PyMuPDF pillow google-genai
!pip install sentence-transformers
!pip install PyMuPDF google-genai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 45.3 MB/s eta 0:00:00


# **cv and job parsing**

In [ ]:
import os
import json
import fitz
from google import genai
from google.genai import types
from google.colab import userdata

try:
    # Load the key securely from Colab Secrets
    os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
    client = genai.Client()
    MODEL_NAME = "gemini-2.5-flash"
    print("Gemini Client initialized successfully using Colab Secrets.")
except Exception as e:
    print(f"Error initializing Gemini Client: {e}")
    print("Please ensure you have set your 'GEMINI_API_KEY' in Colab Secrets.")

Gemini Client initialized successfully using Colab Secrets.


In [ ]:
#Define JSON Schemas for CV and JD ---

# Schema for parsing a Candidate's CV
CV_SCHEMA = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "name": types.Schema(type=types.Type.STRING, description="Full name of the candidate."),
        "email": types.Schema(type=types.Type.STRING, description="Candidate's email address."),
        "phone": types.Schema(type=types.Type.STRING, description="Candidate's phone number."),
        "total_experience_years": types.Schema(
            type=types.Type.NUMBER,
            description="Total years of professional experience (numeric value)."
        ),
        "skills": types.Schema(
            type=types.Type.ARRAY,
            items=types.Schema(type=types.Type.STRING),
            description="A list of core technical and soft skills."
        ),
        "education": types.Schema(
            type=types.Type.ARRAY,
            items=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "degree": types.Schema(type=types.Type.STRING),
                    "institution": types.Schema(type=types.Type.STRING),
                    "year": types.Schema(type=types.Type.STRING)
                }
            )
        ),
        "jobs": types.Schema(
            type=types.Type.ARRAY,
            items=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "title": types.Schema(type=types.Type.STRING),
                    "company": types.Schema(type=types.Type.STRING),
                    "summary": types.Schema(type=types.Type.STRING)
                }
            )
        ),
        "projects": types.Schema(
            type=types.Type.ARRAY,
            items=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "name": types.Schema(type=types.Type.STRING),
                    "description": types.Schema(type=types.Type.STRING),
                    "technologies": types.Schema(
                        type=types.Type.ARRAY,
                        items=types.Schema(type=types.Type.STRING)
                    )
                }
            )
        ),
        "internships": types.Schema(
            type=types.Type.ARRAY,
            items=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "title": types.Schema(type=types.Type.STRING),
                    "company": types.Schema(type=types.Type.STRING),
                    "duration": types.Schema(type=types.Type.STRING),
                    "summary": types.Schema(type=types.Type.STRING)
                }
            )
        )
    }
)


# Schema for parsing a Job Description (JD)
JD_SCHEMA = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "job_title": types.Schema(type=types.Type.STRING, description="The official title of the role being advertised."),
        "company": types.Schema(type=types.Type.STRING, description="The name of the hiring company."),
        "min_experience_years": types.Schema(type=types.Type.NUMBER, description="The minimum required years of experience (numeric value, use 0 if none specified)."),
        "required_skills": types.Schema(type=types.Type.ARRAY, items=types.Schema(type=types.Type.STRING), description="A list of ESSENTIAL technical and soft skills."),
        "preferred_skills": types.Schema(type=types.Type.ARRAY, items=types.Schema(type=types.Type.STRING), description="A list of optional or 'nice-to-have' skills."),
        "qualifications": types.Schema(type=types.Type.ARRAY, items=types.Schema(type=types.Type.STRING), description="Required degrees or certifications (e.g., 'BS in Computer Science')."),
    }
)

#Parsing Function

def parse_document_with_llm(document_path, schema, document_type):
    """
    Reads a document (PDF or Text), extracts text, and uses Gemini
    to parse it into a structured Python dictionary based on the provided schema.
    """
    # 1. Extract Text from Document
    text = ""
    try:
        if document_path.lower().endswith('.pdf'):
            doc = fitz.open(document_path)
            for page in doc:
                text += page.get_text()
        else:
            with open(document_path, 'r', encoding='utf-8') as f:
                text = f.read()

        if not text:
            return {"error": f"Could not extract text from {document_type} file."}
    except Exception as e:
        return {"error": f"{document_type} reading failed: {e}"}

    # 2. Call the Gemini API for structured extraction
    prompt = f"Analyze the following {document_type} text and extract the information based *strictly* on the provided JSON schema. Do not include any text outside of the JSON object.\n\n{document_type} Text:\n---\n{text}"

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=[prompt],
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=schema,
            ),
        )
        # Convert the response JSON string into a Python dictionary
        return json.loads(response.text)

    except Exception as e:
        return {"error": f"Gemini API call or JSON parsing failed: {e}"}




#Execution and Demonstration (You must create these files in Colab) ---

#DUMMY FILE SETUP: In a real run, you must upload a 'my_cv.pdf' and a 'senior_dev_jd.txt'
# to your Colab /content/ folder, or create placeholder files.

# Placeholder File Paths (Change these to your actual uploaded file names)
CV_FILE_PATH = 'yourCV'
JD_FILE_PATH = 'JOB-DESCRIPTION'

#Parse CV
print(f"| A. PARSING CV: {CV_FILE_PATH}")
cv_data = parse_document_with_llm(
    document_path=CV_FILE_PATH,
    schema=CV_SCHEMA,
    document_type="CV/Resume"
)
print(json.dumps(cv_data, indent=2))

#Parse Job Description
print(f"| B. PARSING JOB DESCRIPTION: {JD_FILE_PATH}")
jd_data = parse_document_with_llm(
    document_path=JD_FILE_PATH,
    schema=JD_SCHEMA,
    document_type="JOB DESCRIPTION"
)
print(json.dumps(jd_data, indent=2))


In [ ]:

from sentence_transformers import SentenceTransformer, util

model_sim = SentenceTransformer("all-MiniLM-L6-v2")

def semantic_similarity(a, b):
    emb1 = model_sim.encode(a, convert_to_tensor=True)
    emb2 = model_sim.encode(b, convert_to_tensor=True)
    return util.cos_sim(emb1, emb2).item()

def match_skills_semantically(cv_skills, jd_skills, threshold=0.60):
    matched = []
    for jd_skill in jd_skills:
        for cv_skill in cv_skills:
            score = semantic_similarity(cv_skill, jd_skill)
            if score >= threshold:
                matched.append((cv_skill, jd_skill, round(score, 2)))
                break
    return matched


#Extract data from parsed CV & JD
cv_skills = [s.lower() for s in cv_data.get("skills", [])]
required_skills = [s.lower() for s in jd_data.get("required_skills", [])]
preferred_skills = [s.lower() for s in jd_data.get("preferred_skills", [])]

total_exp = cv_data.get("total_experience_years", 0)
min_exp = jd_data.get("min_experience_years", 0)

# Reset points
points = 0


# Semantic Required Skill Matching
required_matches = match_skills_semantically(cv_skills, required_skills)
if required_skills:
    points += 40 * (len(required_matches) / len(required_skills))


# Semantic Preferred Skill Matching
preferred_matches = match_skills_semantically(cv_skills, preferred_skills)
if preferred_skills:
    points += 20 * (len(preferred_matches) / len(preferred_skills))


# Experience Matching (same formula)
if total_exp >= min_exp:
    points += 40
elif min_exp > 0:
    points += 40 * (total_exp / min_exp)


# Feedback
feedback = (
    f"Experience: {total_exp} years (Min required: {min_exp}).\n"
    f"Semantic required skill matches: {len(required_matches)}/{len(required_skills)}.\n"
    f"Semantic preferred skill matches: {len(preferred_matches)}/{len(preferred_skills)}.\n"
    f"Examples: {required_matches[:3]}"
)

print("Semantic Match Score:", round(points, 2))
print("Feedback:\n", feedback)
